# Day 2: Combinational Building Blocks

## Pre-Class Videos (~45 minutes total)

| # | Segment | Duration | File | Slides |
|---|---------|----------|------|--------|
| 1 | Data Types and Vectors | ~15 min | `d02_s1_data_types_and_vectors.html` | 7 |
| 2 | Operators | ~12 min | `d02_s2_operators.html` | 7 |
| 3 | Sized Literals & Width Matching | ~8 min | `d02_s3_sized_literals_width_matching.html` | 5 |
| 4 | The 7-Segment Display | ~10 min | `d02_s4_seven_segment_display.html` | 8 |

## Code Examples

| File | Description | Synthesizable? |
|------|-------------|----------------|
| `code/day02_ex01_vector_ops.v` | Bit selection, concatenation, replication, sign extension | Yes |
| `code/day02_ex02_mux_2to1.v` | Parameterized 2-to-1 multiplexer | Yes |
| `code/day02_ex03_hex_to_7seg.v` | Complete hex-to-7-segment decoder (active low) | Yes |

## Diagrams

| File | Used In | Description |
|------|---------|-------------|
| `diagrams/d02_wire_vs_reg.svg` | Seg 1 | wire vs reg decision flowchart |
| `diagrams/d02_seven_segment.svg` | Seg 4 | 7-segment display with labeled segments (a-g) |

## Pre-Class Quiz

See `day02_quiz.md` — 4 questions. Also embedded as interactive slides at end of Segment 4.

## Naming Convention

| Pattern | Example |
|---------|---------|
| `d##_s#_topic.html` | `d02_s1_data_types_and_vectors.html` |
| `day##_ex##_name.v` | `day02_ex01_vector_ops.v` |
| `d##_name.svg` | `d02_wire_vs_reg.svg` |
| `day##_support.md` | `day02_quiz.md` |

## Directory Structure

```
week1_day02/
├── day02_readme.md
├── day02_quiz.md
├── d02_s1_data_types_and_vectors.html
├── d02_s2_operators.html
├── d02_s3_sized_literals_width_matching.html
├── d02_s4_seven_segment_display.html
├── code/
│   ├── day02_ex01_vector_ops.v
│   ├── day02_ex02_mux_2to1.v
│   └── day02_ex03_hex_to_7seg.v
└── diagrams/
    ├── d02_wire_vs_reg.svg
    └── d02_seven_segment.svg
```

---
## Code Examples

### `day02_ex01_vector_ops.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day02_ex01_vector_ops.v
// Course:  Accelerated HDL for Digital System Design — Day 2
// Board:   Nandland Go Board (Lattice iCE40 HX1K, VQ100)
//
// Description:
//   Demonstrates vector operations: bit selection, concatenation,
//   replication, and sign extension. Connect switches to i_data[3:0]
//   and observe results on LEDs and 7-segment display.
//
// Build:
//   yosys -p "synth_ice40 -top vector_ops -json day02_ex01_vector_ops.json" day02_ex01_vector_ops.v
//   nextpnr-ice40 --hx1k --package vq100 --pcf go_board.pcf \
//                 --json day02_ex01_vector_ops.json --asc day02_ex01_vector_ops.asc
//   icepack day02_ex01_vector_ops.asc day02_ex01_vector_ops.bin
//   iceprog day02_ex01_vector_ops.bin
//-----------------------------------------------------------------------------
module vector_ops (
    input  wire [7:0] i_data,
    input  wire [3:0] i_nibble_a,
    input  wire [3:0] i_nibble_b,
    output wire [7:0] o_concat,
    output wire [7:0] o_replicate,
    output wire [7:0] o_sign_ext,
    output wire [3:0] o_upper,
    output wire [3:0] o_lower,
    output wire       o_msb
);
    // Bit selection
    assign o_msb   = i_data[7];
    assign o_upper = i_data[7:4];
    assign o_lower = i_data[3:0];

    // Concatenation: join two 4-bit values → one 8-bit value
    assign o_concat = {i_nibble_a, i_nibble_b};

    // Replication: 8 copies of bit 0
    assign o_replicate = {8{i_data[0]}};

    // Sign extension: extend 4-bit signed value to 8 bits
    assign o_sign_ext = {{4{i_nibble_a[3]}}, i_nibble_a};
endmodule
```

### `day02_ex02_mux_2to1.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day02_ex02_mux_2to1.v
// Course:  Accelerated HDL for Digital System Design — Day 2
// Board:   Nandland Go Board (Lattice iCE40 HX1K, VQ100)
//
// Description:
//   Parameterized 2-to-1 multiplexer using the conditional operator.
//   When sel=1, output is a; when sel=0, output is b.
//
// Build:
//   yosys -p "synth_ice40 -top mux_2to1 -json day02_ex02_mux_2to1.json" day02_ex02_mux_2to1.v
//-----------------------------------------------------------------------------
module mux_2to1 #(
    parameter WIDTH = 4
)(
    input  wire [WIDTH-1:0] i_a,
    input  wire [WIDTH-1:0] i_b,
    input  wire             i_sel,
    output wire [WIDTH-1:0] o_y
);
    assign o_y = i_sel ? i_a : i_b;
endmodule
```

### `day02_ex03_hex_to_7seg.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day02_ex03_hex_to_7seg.v
// Course:  Accelerated HDL for Digital System Design — Day 2
// Board:   Nandland Go Board (Lattice iCE40 HX1K, VQ100)
//
// Description:
//   Hexadecimal to 7-segment decoder. Maps 4-bit input (0-F) to
//   active-low segment outputs {a,b,c,d,e,f,g} for Go Board display.
//   0 = segment ON, 1 = segment OFF.
//
// Build:
//   yosys -p "synth_ice40 -top hex_to_7seg -json day02_ex03_hex_to_7seg.json" day02_ex03_hex_to_7seg.v
//-----------------------------------------------------------------------------
module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg   // {a,b,c,d,e,f,g} active low
);
    always @(*) begin
        case (i_hex)       //     abcdefg
            4'h0: o_seg = 7'b0000001;
            4'h1: o_seg = 7'b1001111;
            4'h2: o_seg = 7'b0010010;
            4'h3: o_seg = 7'b0000110;
            4'h4: o_seg = 7'b1001100;
            4'h5: o_seg = 7'b0100100;
            4'h6: o_seg = 7'b0100000;
            4'h7: o_seg = 7'b0001111;
            4'h8: o_seg = 7'b0000000;
            4'h9: o_seg = 7'b0000100;
            4'hA: o_seg = 7'b0001000;
            4'hB: o_seg = 7'b1100000;
            4'hC: o_seg = 7'b0110001;
            4'hD: o_seg = 7'b1000010;
            4'hE: o_seg = 7'b0110000;
            4'hF: o_seg = 7'b0111000;
            default: o_seg = 7'b1111111;
        endcase
    end
endmodule
```

---
## Pre-Class Self-Check Quiz

**Q1:** Does `reg` always synthesize to a register?

<details><summary>Answer</summary>
No. `reg` in `always @(*)` = combinational logic. `reg` in `always @(posedge clk)` = flip-flop. The keyword only means "can be assigned in a procedural block."
</details>

**Q2:** What is `4'hF` in binary? How many bits?

<details><summary>Answer</summary>
4-bit binary `1111` (decimal 15). The `4'` specifies 4 bits, `h` means hexadecimal.
</details>

**Q3:** What does `assign y = sel ? a : b;` synthesize to?

<details><summary>Answer</summary>
A 2-to-1 multiplexer. When `sel=1`, output is `a`. When `sel=0`, output is `b`.
</details>

**Q4:** What's the difference between `&` and `&&`?

<details><summary>Answer</summary>
`&` is bitwise AND (per-bit, result same width). `&&` is logical AND (1-bit true/false result). `4'b1010 & 4'b0101 = 4'b0000` but `4'b1010 && 4'b0101 = 1'b1`.
</details>